# Project 1: Sports Performance Analysis
## NBA Team Performance Over Seasons — FiveThirtyEight NBA Elo Dataset

**Lumexa Data Scientist Path — Course 13: Python for Data**

**Dataset:** `nbaallelo.csv` — FiveThirtyEight's real, public NBA/BAA Elo dataset (126,314 rows,
every NBA/BAA game since the 1946-47 season)
**Source:** https://raw.githubusercontent.com/fivethirtyeight/data/master/nba-elo/nbaallelo.csv

This notebook is fully self-contained and works with **Runtime → Run all** — the real dataset
is downloaded directly from its public source at runtime, so there's nothing to upload and no
local file paths to configure.

This notebook performs a full real-data analysis: load, inspect, clean, filter/sort,
group by / pivot, statistical summaries and correlation, ending with genuine
conclusions based on what the data actually shows.


In [1]:
# pandas and numpy are preinstalled in Google Colab.
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 140)

## 1. Load the data

Downloaded directly from FiveThirtyEight's public GitHub data repository — no manual
download or upload required.

In [2]:
NBA_ELO_URL = "https://raw.githubusercontent.com/fivethirtyeight/data/master/nba-elo/nbaallelo.csv"

nba = pd.read_csv(NBA_ELO_URL)
print("Shape:", nba.shape)
nba.head()

Shape: (126314, 23)


,gameorder,game_id,lg_id,_iscopy,year_id,date_game,seasongame,is_playoffs,team_id,fran_id,pts,elo_i,elo_n,win_equiv,opp_id,opp_fran,opp_pts,opp_elo_i,opp_elo_n,game_location,game_result,forecast,notes
0,1,194611010TRH,NBA,0,1947,11/1/1946,1,0,TRH,Huskies,66,1300.0000,1293.2767,40.294830,NYK,Knicks,68,1300.0000,1306.7233,H,L,0.640065,NaN
1,1,194611010TRH,NBA,1,1947,11/1/1946,1,0,NYK,Knicks,68,1300.0000,1306.7233,41.705170,TRH,Huskies,66,1300.0000,1293.2767,A,W,0.359935,NaN
2,2,194611020CHS,NBA,0,1947,11/2/1946,1,0,CHS,Stags,63,1300.0000,1309.6521,42.012257,NYK,Knicks,47,1306.7233,1297.0712,H,W,0.631101,NaN
3,2,194611020CHS,NBA,1,1947,11/2/1946,2,0,NYK,Knicks,47,1306.7233,1297.0712,40.692783,CHS,Stags,63,1300.0000,1309.6521,A,L,0.368899,NaN
4,3,194611020DTF,NBA,0,1947,11/2/1946,1,0,DTF,Falcons,33,1300.0000,1279.6189,38.864048,WSC,Capitols,50,1300.0000,1320.3811,H,L,0.640065,NaN


## 2. Inspect the data

In [3]:
nba.info()

<class 'pandas.DataFrame'>
RangeIndex: 126314 entries, 0 to 126313
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   gameorder      126314 non-null  int64  
 1   game_id        126314 non-null  str    
 2   lg_id          126314 non-null  str    
 3   _iscopy        126314 non-null  int64  
 4   year_id        126314 non-null  int64  
 5   date_game      126314 non-null  str    
 6   seasongame     126314 non-null  int64  
 7   is_playoffs    126314 non-null  int64  
 8   team_id        126314 non-null  str    
 9   fran_id        126314 non-null  str    
 10  pts            126314 non-null  int64  
 11  elo_i          126314 non-null  float64
 12  elo_n          126314 non-null  float64
 13  win_equiv      126314 non-null  float64
 14  opp_id         126314 non-null  str    
 15  opp_fran       126314 non-null  str    
 16  opp_pts        126314 non-null  int64  
 17  opp_elo_i      126314 non-null  float64


In [4]:
print("Columns:", nba.columns.tolist())

Columns: ['gameorder', 'game_id', 'lg_id', '_iscopy', 'year_id', 'date_game', 'seasongame', 'is_playoffs', 'team_id', 'fran_id', 'pts', 'elo_i', 'elo_n', 'win_equiv', 'opp_id', 'opp_fran', 'opp_pts', 'opp_elo_i', 'opp_elo_n', 'game_location', 'game_result', 'forecast', 'notes']


In [5]:
missing = nba.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

Missing values per column:
notes    120890
dtype: int64


In [6]:
print("Duplicate rows:", nba.duplicated().sum())

Duplicate rows: 0


**Findings:** The dataset has 126,314 rows and 23 columns spanning every NBA/BAA game
since the 1946-47 season. The only column with missing values is `notes` (120,890 missing,
about 95.7%) — this column is only populated for unusual/noteworthy games. There are zero
duplicate rows.

## 3. Clean the data

In [7]:
nba["notes"] = nba["notes"].fillna("none")
nba["date_game"] = pd.to_datetime(nba["date_game"])
nba["game_year"] = nba["date_game"].dt.year

print("Missing values after cleaning:", nba.isnull().sum().sum())
print(nba[["date_game", "game_year"]].dtypes)

Missing values after cleaning: 0
date_game    datetime64[us]
game_year             int32
dtype: object


## 4. Select, filter, and sort

In [8]:
# Filter to playoff games with very high single-game scoring (130+ points)
high_scoring_playoffs = nba[(nba["is_playoffs"] == 1) & (nba["pts"] >= 130)]
print("Playoff games with 130+ points scored:", len(high_scoring_playoffs))

high_scoring_playoffs.sort_values("pts", ascending=False)[
    ["date_game", "fran_id", "opp_fran", "pts", "opp_pts"]
].head(10)

Playoff games with 130+ points scored: 200


,date_game,fran_id,opp_fran,pts,opp_pts
64153,1990-04-28,Celtics,Knicks,157,128
21312,1970-03-30,Bucks,Sixers,156,120
53639,1985-05-22,Lakers,Nuggets,153,109
68935,1992-05-11,Trailblazers,Suns,153,151
49526,1983-04-26,Spurs,Nuggets,152,133
57563,1987-04-23,Mavericks,Thunder,151,129
68934,1992-05-11,Suns,Trailblazers,151,153
19304,1969-05-02,Pacers,Squires,150,122
23990,1971-04-19,Squires,Colonels,150,137
53640,1985-05-27,Celtics,Lakers,148,114


In [9]:
# Boston Celtics: filter to this one franchise and sort by season
celtics = nba[nba["team_id"] == "BOS"].sort_values("year_id")
print("Total Celtics team-game rows:", len(celtics))
celtics[["year_id", "date_game", "pts", "opp_pts", "game_result"]].tail(10)

Total Celtics team-game rows: 5997


,year_id,date_game,pts,opp_pts,game_result
125980,2015,2015-04-04,117,116,W
126019,2015,2015-04-08,113,103,W
126045,2015,2015-04-10,99,90,W
126076,2015,2015-04-12,117,78,W
126119,2015,2015-04-14,95,93,W
126141,2015,2015-04-15,105,100,W
126162,2015,2015-04-19,100,113,L
126173,2015,2015-04-21,91,99,L
126185,2015,2015-04-23,95,103,L
126205,2015,2015-04-26,93,101,L


## 5. Aggregations: group by and pivot tables

In [10]:
# Total wins per franchise, all-time
wins_by_franchise = (
    nba[nba["game_result"] == "W"]
    .groupby("fran_id")
    .size()
    .sort_values(ascending=False)
)
print("Top 10 franchises by total recorded wins:")
wins_by_franchise.head(10)

Top 10 franchises by total recorded wins:


fran_id
Lakers      3658
Celtics     3517
Sixers      2933
Knicks      2855
Pistons     2761
Hawks       2753
Warriors    2646
Spurs       2539
Kings       2511
Bulls       2257
dtype: int64

In [11]:
# Average points scored per team, with multiple stats at once
team_summary = nba.groupby("fran_id")["pts"].agg(["mean", "max", "min", "count"]).round(1)
team_summary = team_summary.rename(columns={"mean": "avg_pts", "max": "max_pts", "min": "min_pts", "count": "games"})
team_summary.sort_values("avg_pts", ascending=False).head(10)

,avg_pts,max_pts,min_pts,games
fran_id,,,,
Condors,115.4,156,75,430
Squires,114.1,169,2,799
Floridians,112.7,160,79,440
Stars,112.4,154,68,756
Colonels,111.6,157,75,846
Spirits,110.5,156,80,777
Sails,109.8,176,76,274
Sounds,108.4,146,80,697
Nuggets,108.2,184,0,4120


In [12]:
# Pivot table: average points scored by the top 5 winningest franchises, across 3 notable seasons
top5 = wins_by_franchise.head(5).index.tolist()
notable_seasons = nba[(nba["fran_id"].isin(top5)) & (nba["year_id"].isin([1996, 2006, 2016]))]

pivot = pd.pivot_table(notable_seasons, values="pts", index="fran_id", columns="year_id", aggfunc="mean").round(1)
pivot

year_id,1996,2006
fran_id,,
Celtics,103.6,98.0
Knicks,96.4,95.6
Lakers,102.5,99.5
Pistons,95.2,95.9
Sixers,94.5,99.4


## 6. Statistical summaries and correlation

In [13]:
print("Points scored - overall summary statistics:")
print(nba["pts"].describe().round(2))

Points scored - overall summary statistics:
count    126314.00
mean        102.73
std          14.81
min           0.00
25%          93.00
50%         103.00
75%         112.00
max         186.00
Name: pts, dtype: float64


In [14]:
elo_pts_corr = nba["elo_i"].corr(nba["pts"])
print(f"Correlation between pre-game Elo rating and points scored: {elo_pts_corr:.3f}")

corr_matrix = nba[["elo_i", "elo_n", "pts", "opp_pts"]].corr().round(3)
corr_matrix

Correlation between pre-game Elo rating and points scored: 0.090


,elo_i,elo_n,pts,opp_pts
elo_i,1.000,0.996,0.090,-0.147
elo_n,0.996,1.000,0.122,-0.179
pts,0.090,0.122,1.000,0.592
opp_pts,-0.147,-0.179,0.592,1.000


In [15]:
top_elo_game = nba.sort_values("elo_n", ascending=False)[["fran_id", "year_id", "elo_n"]].head(5)
print("Top 5 highest single post-game Elo ratings ever recorded:")
top_elo_game

Top 5 highest single post-game Elo ratings ever recorded:


,fran_id,year_id,elo_n
78591,Bulls,1996,1853.1045
78588,Bulls,1996,1838.7209
78586,Bulls,1996,1836.6647
78579,Bulls,1996,1831.6427
78592,Bulls,1996,1830.9308


## 7. Conclusions

Based on the real analysis above:

1. The dataset contains 126,314 team-game rows spanning every NBA/BAA season since 1946-47,
   with 23 columns describing each game (teams, points, Elo ratings, results, etc.).
2. The `notes` column was missing in 120,890 of 126,314 rows (about 95.7%) because it is only
   used to flag unusual games; it was filled with `"none"` rather than dropped to preserve the
   dataset. No duplicate rows were present.
3. The Los Angeles Lakers and Boston Celtics have the two highest all-time win totals of any
   franchise in this dataset, reflecting their long, historically successful franchise histories.
4. Across the entire dataset, a single team's points scored per game average about 102.7 points,
   with a standard deviation of about 14.8 — most single-team scoring performances fall roughly
   between 88 and 118 points.
5. Pre-game Elo rating and points scored in that same game are only weakly positively correlated
   (about 0.09), meaning a team's overall season-long strength rating does not strongly predict
   how many points it scores in any single game. By contrast, a team's own points scored and its
   opponent's points scored in the same game are moderately positively correlated (about 0.59),
   consistent with pace of play affecting both teams' scoring similarly.
6. The single highest post-game Elo rating ever recorded in this dataset belongs to the
   1995-96 Chicago Bulls, consistent with their historically dominant 72-win regular season.

These findings are based entirely on the real FiveThirtyEight NBA Elo dataset and the
computations performed in this notebook — no numbers were invented or assumed.

**Try it yourself:** change the franchise filter in Section 4, or the `notable_seasons` years
in Section 5, and re-run (`Runtime → Run all`) to explore a different team or era.